# Initial Datahandeling

### This notebook shows the initial handling of utterances in the Danish Parliament.
### It includes the following:

- Loading the two overlapping datasets of utterences in the Danish Parliament
    - Due to the ParlLawSpeech data being in .rds format, it was read into r, converted to .csv format and then saved in this repo
- Merging them by end data of corp_v2
- Sentence segmentation
- Reformatting to json object for easier handling during labeling for validation, training and inference
- Datacleaning
- Splitting into training data (including validationset to be extracted), and inference data

In [1]:
# Load packages

import pandas as pd
import os
from danish_minister_party_pipeline import assign_party


# Corp_Folketing_V2

In [2]:
#read the corp folketing 

'''corp_v2_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "data",
                            "csv_meetings",
                            "Corp_Folketing_V2.csv")'''

corp_v2_path = os.path.join("..",
                            "..",
                            "raw_data",
                            "Corp_Folketing_V2.csv")

# Read the two datasets
df_corp = pd.read_csv(corp_v2_path, index_col=0)
#x = df_corp.pop("Unnamed: 0") #dummy variable

#make date a datetime object
df_corp["date"] = pd.to_datetime(df_corp["date"])

In [3]:
df_corp.head()

,date,agenda,speechnumber,speaker,party,party.facts.id,chair,terms,text,parliament,iso3country
0,1997-10-07,Dagsorden,1,Gert Petersen,NaN,NaN,True,191,Mødet er åbnet. I henhold til grundloven er Fo...,DK-Folketing,DNK
1,1997-10-07,Dagsorden,2,Formanden,NaN,NaN,True,182,"Jeg vil gerne takke Tinget for den tillid, man...",DK-Folketing,DNK
2,1997-10-07,Statsministerens redegørelse i henhold til gru...,3,Poul Nyrup Rasmussen,S,379.0,False,18662,For 25 år siden sagde et flertal i befolkninge...,DK-Folketing,DNK
3,1997-10-09,1) Indstilling fra Udvalget til Valgs Prøvelse.,2,Formanden,NaN,NaN,True,47,Fra Udvalget til Valgs Prøvelse har jeg modtag...,DK-Folketing,DNK
4,1997-10-09,2) Forhandling om redegørelse nr. R 1.,3,Torben Lund,S,379.0,False,2865,Vi står over for en meget afgørende folketings...,DK-Folketing,DNK


In [3]:
#This dataset spans the following time period

min_time_corp = df_corp["date"].min()
max_time_corp = df_corp["date"].max()

print(f"The Corp dataset has coverage from {min_time_corp.date()} to {max_time_corp.date()}")

The Corp dataset has coverage from 1997-10-07 to 2018-12-20


# XML speeches

In [4]:
#read the PLS data. 
# OBS has been made to .csv in file ./PLS_rds_to_csv.RMD

'''xml_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "data",
                            "csv_meetings",
                            "meetings.csv")'''

xml_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "Data_outside_git",
                            "meetings.csv")

# Read the two datasets
df_xml = pd.read_csv(xml_path)

#make date a datetime object
df_xml["date"] = pd.to_datetime(df_xml["date"])

In [5]:
df_xml.head()

,paragraph_nr,date,speaker,party,role,text,source_file
0,2,2009-12-09,Sophie Hæstorp Andersen,S,medlem,"Tak. Den 17. juni 2009, altså i år, fastslog R...",20091_M29_helemoedet.xml
1,4,2009-12-09,Jakob Axel Nielsen,NaN,minister,Tusind tak. Oppositionen har bedt mig redegøre...,20091_M29_helemoedet.xml
2,6,2009-12-09,Sophie Hæstorp Andersen,S,medlem,Socialdemokratiet går ind for et sundhedsvæsen...,20091_M29_helemoedet.xml
3,8,2009-12-09,Birgitte Josefsen,V,medlem,Jeg står med et notat fra Amtsrådsforeningen d...,20091_M29_helemoedet.xml
4,10,2009-12-09,Sophie Hæstorp Andersen,S,medlem,"Det er jo en meget besnærende tanke, som reger...",20091_M29_helemoedet.xml


In [6]:
#This dataset spans the following time period

min_time_xml = df_xml["date"].min()
max_time_xml = df_xml["date"].max()

print(f"The xml dataset has coverage from {min_time_xml.date()} to {max_time_xml.date()}")

The xml dataset has coverage from 2009-10-06 to 2026-02-26


# Make the XML speeches match the format of ParlSpeech

To do so, the following must be done:

- Chair column as TRUE/FALSE - DONE
- Missing values for all non-chairmen to be inferred
- Make sure the two datasets has matching column structure



In [7]:
# Keep only rows in df2 strictly after max date of df1
df_xml_filtered = df_xml[df_xml['date'] > max_time_corp]

no_party = df_xml_filtered.loc[df_xml_filtered["party"].isna(), "speaker"]
no_party_unique = no_party.unique()

print(no_party_unique) # Print the names of the speakers that have no party affiliation
print(len(no_party))    # print the expected number of rows to be modified

['Nicolai Wammen' 'Lars Aagaard' 'Magnus Heunicke' 'Marie Bjerre'
 'Ane Halsboe-Jørgensen' 'Louise Schack Elholm' 'Morten Bødskov'
 'Kaare Dybvad Bek' 'Mette Frederiksen' 'Jakob Engel-Schmidt'
 'Jeppe Bruus' 'Jacob Jensen' 'Mette Kierkgaard' 'Sophie Løhde'
 'Mattias Tesfaye' 'Thomas Danielsen' ' ' 'Lars Løkke Rasmussen'
 'Peter Hummelgaard' 'Pernille Rosenkrantz-Theil' 'Christina Egelund'
 'Troels Lund Poulsen' 'Jakob Ellemann-Jensen' 'Dan Jørgensen'
 'Sophie Hæstorp Andersen' 'Torsten Schack Pedersen' 'Rasmus Stoklund'
 'Morten Dahlin' 'Caroline Stage Olsen' 'Stephanie Lose' 'Jeppe Kofod'
 'Rasmus Prehn' 'Lea Wermelin' 'Benny Engelbrecht' 'Nick Hækkerup'
 'Trine Bramsen' 'Jesper Petersen' 'Astrid Krag' 'Simon Kollerup'
 'Flemming Møller Mortensen' 'Christian Rabjerg Madsen' 'Joy Mogensen'
 'Mogens Jensen' 'Kaare Dybvad' 'Peter Hummelgaard Thomsen'
 'Rasmus Jarlov' 'Søren Pape Poulsen' 'Inger Støjberg' 'Ellen Trane Nørby'
 'Kristian Jensen' 'Simon Emil Ammitzbøll-Bille'
 'Lars Christia

In [ ]:
df_xml_w_parties = assign_party(df_xml_filtered)

[assign_party]  Total rows          : 180111
[assign_party]  Pre-filled (skipped): 151314
[assign_party]  Newly matched       : 28350
[assign_party]  Unmatched / empty   : 0


In [10]:
#check if all are filled
no_party = df_xml_w_parties.loc[df_xml_w_parties["party"].isna(), "speaker"]
no_party_unique = no_party.unique()

print(no_party_unique) # Print the names of the speakers that have no party affiliation
print(len(no_party)) 

[' ']
447


# OBS there are 447 paragraphs witout speaker or party. Why?

See header here for the fist 10

In [14]:
df_xml_w_parties.loc[df_xml_w_parties["party"].isna()]

,paragraph_nr,date,speaker,party,role,text,source_file
32593,7,2023-03-14,,NaN,NaN,(Punktet er udgået af dagsordenen).,20222_M32_helemoedet.xml
35455,157,2023-02-08,,NaN,NaN,"(Spørgsmålet er udgået, da det er taget tilbag...",20222_M24_helemoedet.xml
35952,60,2023-02-21,,NaN,NaN,Tak. Fru Lisbeth Bech-Nielsen.,20222_M26_helemoedet.xml
37056,1,2023-02-23,,NaN,NaN,(Punktet er udgået af dagsordenen).,20222_M28_helemoedet.xml
38752,5,2022-11-15,,NaN,NaN,Fra statsministeren har jeg modtaget følgende ...,20222_M1_helemoedet.xml
...,...,...,...,...,...,...,...
390400,254,2024-05-14,,NaN,NaN,(Punktet er udgået af dagsordenen).,20231_M92_helemoedet.xml
390401,255,2024-05-14,,NaN,NaN,(Punktet er udgået af dagsordenen).,20231_M92_helemoedet.xml
390402,256,2024-05-14,,NaN,NaN,(Punktet er udgået af dagsordenen).,20231_M92_helemoedet.xml
390420,19,2024-04-17,,NaN,NaN,(Spørgsmålet er udgået af dagsordenen).,20231_M80_helemoedet.xml


In [17]:
#dropping irrelevant columns before merge
df_xml_w_parties.drop(columns=["paragraph_nr", "role", "source_file"])

,date,speaker,party,text
29467,2023-05-15,Benny Engelbrecht,S,"Tak, formand. Ikke mindst vil jeg også sige ta..."
29468,2023-05-15,Dennis Flydtkjær,DD,Jeg vil gerne drøfte den udfordring med hr. Be...
29469,2023-05-15,Benny Engelbrecht,S,Tak til hr. Dennis Flydtkjær for det yderst re...
29470,2023-05-15,Dennis Flydtkjær,DD,Det er meget korrekt. Vi vil meget gerne indgå...
29471,2023-05-15,Benny Engelbrecht,S,"Det er klart, at vi i forbindelse med den fina..."
...,...,...,...,...
391475,2024-01-25,Søren Søndergaard,EL,"Det forstod jeg simpelt hen ikke, og da hr. Ch..."
391476,2024-01-25,Gunvor Wibroe,S,Jeg vil lige sige tak for en fantastisk dunder...
391477,2024-01-25,Søren Søndergaard,EL,"Jeg betvivler overhovedet ikke, at man kan bru..."
391478,2024-01-25,Gunvor Wibroe,S,"Fra Socialdemokratiets side synes vi, at det e..."


# Merging datasets

In [20]:
#rename column(s) in each to match the counterpart

print(df_xml.columns)
print(df_corp.columns) # party.facts.id to partyfacts_ID

Index(['paragraph_nr', 'date', 'speaker', 'party', 'role', 'text',
       'source_file', 'chair'],
      dtype='str')
Index(['date', 'agenda', 'speechnumber', 'speaker', 'party', 'chair', 'text'], dtype='str')


In [18]:
# Dropping irrelevant columnds before merge

df_corp = df_corp.drop(columns=["party.facts.id", "parliament", "iso3country", "terms", "agenda", "speechnumber"])
print(df_corp.columns) #double check

Index(['date', 'speaker', 'party', 'chair', 'text'], dtype='object')


In [22]:

# Keep only rows in df2 strictly after max date of df1
df_xml_filtered = df_xml[df_xml['date'] > max_time_corp]

# Concatenate
df_merged = pd.concat([df_corp, df_xml_filtered], ignore_index=True)

# sort by date
df_merged = df_merged.sort_values('date').reset_index(drop=True)

In [23]:
#check that the span now is the entire period
#This dataset spans the following time period

min_time_merged = df_merged["date"].min()
max_time_merged = df_merged["date"].max()

print(f"The merged datasets has coverage from {min_time_merged.date()} to {max_time_merged.date()}")

The merged datasets has coverage from 1997-10-07 to 2026-02-26


### As the future analysis will discard any utterences from the chair of parliament, as these serve to aid order and time of speaking, chair == True will be discarded now for efficiency reasons. Additionally, they typically speak for a very long time at the beginning fo each recorded session

In [25]:

df_merged = df_merged[df_merged['chair'] == False] # Only include chair == False

'''

df_merged = df_merged.loc[
    (df_merged["chair"].isna()) | ((df_merged["chair"] == "False"))]

df_merged = df_merged[df_merged['role'] != "formand"] # Same idea, formatted differently in the xml dataset
'''


'\n\ndf_merged = df_merged.loc[\n    (df_merged["chair"].isna()) | ((df_merged["chair"] == "False"))]\n\ndf_merged = df_merged[df_merged[\'role\'] != "formand"] # Same idea, formatted differently in the xml dataset\n'

In [26]:
len(df_merged)

639400

In [27]:
df_merged

,date,agenda,speechnumber,speaker,party,chair,text,paragraph_nr,role,source_file
2,1997-10-07,Statsministerens redegørelse i henhold til gru...,3.0,Poul Nyrup Rasmussen,S,False,For 25 år siden sagde et flertal i befolkninge...,NaN,NaN,NaN
3,1997-10-09,2) Forhandling om redegørelse nr. R 1.,136.0,Frank Dahlgaard,KF,False,»De Konservative kræver den sorte skole tilbag...,NaN,NaN,NaN
4,1997-10-09,2) Forhandling om redegørelse nr. R 1.,137.0,Kristian Thulesen Dahl,DF,False,Jeg må give hr. Holger K. Nielsen fuldstændig ...,NaN,NaN,NaN
5,1997-10-09,2) Forhandling om redegørelse nr. R 1.,138.0,Holger K. Nielsen,SF,False,"Til hr. Frank Dahlgaard kan jeg sige, at når m...",NaN,NaN,NaN
6,1997-10-09,2) Forhandling om redegørelse nr. R 1.,139.0,Frank Dahlgaard,KF,False,"Det er jo forkert, hvad hr. Holger K. Nielsen ...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1146394,2026-02-26,NaN,NaN,Rasmus Jarlov,KF,False,"Mener Socialdemokratiet, at man gør noget godt...",76.0,medlem,20251_M57_helemoedet.xml
1146396,2026-02-26,NaN,NaN,Benny Engelbrecht,S,False,"Mener hr. Rasmus Jarlov, at Konservative, den ...",78.0,medlem,20251_M57_helemoedet.xml
1146398,2026-02-26,NaN,NaN,Rasmus Jarlov,KF,False,"Jeg synes ikke, det burde være så svært at for...",80.0,medlem,20251_M57_helemoedet.xml
1146400,2026-02-26,NaN,NaN,Benny Engelbrecht,S,False,"Vi kan ikke vide, præcis hvor omkostningsfuldt...",82.0,medlem,20251_M57_helemoedet.xml


In [29]:
#Save the data

merged_data_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "data",
                            "csv_meetings"
                            "merged_speech_data.csv")
                                
df_merged.to_csv(merged_data_path, index=False)